In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold, chi2, f_classif, mutual_info_classif
from scipy.stats import f_oneway, chi2_contingency

from src.preprocessing import (
    calculate_mean, calculate_median, calculate_mode, calculate_variance, calculate_std,
    calculate_min_max_range, get_missing_summary, ScratchImputer, identify_and_drop_duplicates,
    clean_inconsistent_categories, ScratchLabelEncoder, ScratchOneHotEncoder,
    ScratchIQROutlierDetector, ScratchZScoreDetector, log_transform, ScratchMinMaxScaler,
    ScratchStandardScaler, scratch_train_test_split
)
from src.feature_selection import (
    calculate_feature_variances, scratch_correlation_matrix,
    scratch_chi2_test, scratch_anova_f_test, calculate_mutual_information
)

In [ ]:
### Step 1: Dataset Generation / Ingestion
# Generating synthetic benchmark dataset satisfying assignment criteria

np.random.seed(42)
n_samples = 300

ages = np.random.randint(18, 70, size=n_samples).astype(float)
ages[np.random.choice(n_samples, 15, replace=False)] = np.nan # Missing

incomes = np.random.normal(50000, 15000, size=n_samples).round(-2)
incomes[np.random.choice(n_samples, 20, replace=False)] = np.nan # Missing
incomes[0] = 350000.0 # Outlier

credit_scores = np.random.randint(300, 850, size=n_samples)
zero_var_col = np.ones(n_samples) * 10.0 # Zero variance

genders = np.random.choice(['Male', 'Female', 'MALE', 'female', 'M'], size=n_samples)
cities = np.random.choice(['Kanpur', 'Delhi', 'Lucknow'], size=n_samples)
purchased = np.random.choice(['No', 'Yes'], size=n_samples, p=[0.6, 0.4])

df_raw = pd.DataFrame({
    'Age': ages,
    'Income': incomes,
    'CreditScore': credit_scores,
    'ConstantFeature': zero_var_col,
    'Gender': genders,
    'City': cities,
    'Purchased': purchased
})
# Append 5 duplicate rows
df_raw = pd.concat([df_raw, df_raw.iloc[:5]], ignore_index=True)
df_raw.to_csv('../dataset/dataset.csv', index=False)
print("Initial Dataset Shape:", df_raw.shape)

In [ ]:
### Step 2: Initial Exploration & Baseline Stats (Part B)

print("Missing Summary Before Imputation:")
print(get_missing_summary(df_raw))

age_vals = df_raw['Age'].tolist()
print("\n--- Descriptive Statistics from Scratch (Age) ---")
print("Mean:", calculate_mean(age_vals))
print("Median:", calculate_median(age_vals))
print("Variance:", calculate_variance(age_vals))
print("Std Dev:", calculate_std(age_vals))
print("Min, Max, Range:", calculate_min_max_range(age_vals))

# Library Verification
print("\n--- Library Verification (Pandas) ---")
print("Pandas Mean:", df_raw['Age'].mean())
print("Pandas Median:", df_raw['Age'].median())
print("Pandas Std:", df_raw['Age'].std())

In [ ]:
### Step 3: Train-Test Split (Part J & K: Avoid Data Leakage)

# Train-test split is executed BEFORE fitting preprocessing parameters
df_train, df_test = scratch_train_test_split(df_raw, test_size=0.2, random_state=42)
print(f"Train samples: {len(df_train)}, Test samples: {len(df_test)}")

In [ ]:
### Step 4: Duplicate & Inconsistent Data Handling (Part D)

df_train, train_dupes = identify_and_drop_duplicates(df_train)
df_test, test_dupes = identify_and_drop_duplicates(df_test)
print(f"Dropped {train_dupes} training duplicates and {test_dupes} testing duplicates.")

gender_clean_map = {'Male': 'Male', 'Female': 'Female', 'M': 'Male'}
df_train = clean_inconsistent_categories(df_train, 'Gender', gender_clean_map)
df_test = clean_inconsistent_categories(df_test, 'Gender', gender_clean_map)

In [ ]:
### Step 5: Missing Value Imputation (Part C)

# Learn parameters on train only
num_cols = ['Age', 'Income']
imputer = ScratchImputer(strategy='median')
imputer.fit(df_train, num_cols)

df_train = imputer.transform(df_train)
df_test = imputer.transform(df_test)

# Verify no null values remain
assert df_train['Age'].isnull().sum() == 0
assert df_test['Income'].isnull().sum() == 0
print("Imputation completed successfully using training statistics.")

In [ ]:
### Step 6: Outlier Detection and Capping (Part F)

iqr_detector = ScratchIQROutlierDetector()
iqr_detector.fit(df_train, ['Income'])
print("Income IQR Bounds:", iqr_detector.bounds_['Income'])

# Cap outliers
df_train = iqr_detector.cap(df_train)
df_test = iqr_detector.cap(df_test)

In [ ]:
### Step 7: Categorical Encoding (Part E)

# Label encode binary target
target_encoder = ScratchLabelEncoder()
target_encoder.fit(df_train['Purchased'])
df_train['Purchased'] = target_encoder.transform(df_train['Purchased'])
df_test['Purchased'] = target_encoder.transform(df_test['Purchased'])

# One-Hot encode nominal features
ohe = ScratchOneHotEncoder()
ohe.fit(df_train, ['Gender', 'City'])
df_train = ohe.transform(df_train)
df_test = ohe.transform(df_test)

In [ ]:
### Step 8: Scaling & Normalization (Part H)

scale_cols = ['Age', 'Income', 'CreditScore']

# Standardize
scaler = ScratchStandardScaler()
scaler.fit(df_train, scale_cols)
df_train_scaled = scaler.transform(df_train)
df_test_scaled = scaler.transform(df_test)

# Library Verification
sk_scaler = StandardScaler()
sk_scaler.fit(df_train[scale_cols])
sk_transformed = sk_scaler.transform(df_train[scale_cols])
print("Max absolute difference in Standardization (Scratch vs Sklearn):",
      np.max(np.abs(df_train_scaled[scale_cols].values - sk_transformed)))

In [ ]:
### Step 9: Feature Selection (Part M)

# M1: Variance Threshold
num_features = ['Age', 'Income', 'CreditScore', 'ConstantFeature']
selected_vars, dropped_vars, variances = apply_variance_threshold(
    df_train, df_test, num_features, threshold=0.01
)
print("Variance Analysis:", variances)
print("Dropped due to low variance:", dropped_vars)

# M2: Pearson Correlation
corr_matrix = scratch_correlation_matrix(df_train, ['Age', 'Income', 'CreditScore'])
print("\n--- Correlation Matrix (Scratch) ---")
print(corr_matrix)

# M3: Chi-Square Test (Discrete City vs Purchased)
chi2_stat, df_val, _ = scratch_chi2_test(df_raw['City'], df_raw['Purchased'])
print(f"\nChi-Square Statistic: {chi2_stat:.4f}, Degrees of Freedom: {df_val}")

# M4: ANOVA F-Test (Age vs Purchased)
f_stat, df_b, df_w = scratch_anova_f_test(df_train['Age'].tolist(), df_train['Purchased'].tolist())
print(f"ANOVA F-Statistic (Age vs Purchased): {f_stat:.4f} (df: {df_b}, {df_w})")

# M5: Mutual Information
mi_val = calculate_mutual_information(df_raw['City'].tolist(), df_raw['Purchased'].tolist())
print(f"Mutual Information (City; Purchased): {mi_val:.4f} bits")

In [ ]:
### Step 10: Visualizations & Distributions (Part I)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df_train['Income'], kde=True, ax=axs[0], color='royalblue')
axs[0].set_title('Income Distribution (After Capping)')
axs[0].set_xlabel('Income')
axs[0].set_ylabel('Frequency')

sns.heatmap(corr_matrix.astype(float), annot=True, cmap='coolwarm', ax=axs[1])
axs[1].set_title('Feature Correlation Heatmap')

plt.tight_layout()
plt.savefig('../results/graphs/preprocessing_visuals.png')
plt.show()
print("Saved visualization to results/graphs/preprocessing_visuals.png")